<h1>电商用户行为画像与转化漏斗分析</h1><blockquote><p><strong>一句话项目简介</strong>：基于十万级电商用户行为数据，使用 Python（Pandas/Matplotlib）与 SQL（SQLite）完成用户画像分层与全链路转化漏斗分析，定位核心流失环节并输出可落地的运营优化建议。</p></blockquote><h2>一、项目概述</h2><h3>1.1 项目背景</h3><p>电商平台的增长核心在于「流量 → 转化」的效率。运营团队面临的典型问题是：<strong>预算该投向哪个渠道？用户在哪个环节流失最多？哪类人群最值得精细化运营？</strong> 本项目以真实电商用户行为数据为对象，通过数据分析回答上述三个业务问题。</p><h3>1.2 数据来源与工具</h3><ul><li><p><strong>数据来源</strong>：和鲸社区公开电商用户行为数据集（CSV 格式，含用户属性与各页面访问行为字段）</p></li><li><p><strong>分析工具</strong>：Python（Pandas / NumPy / Matplotlib）、SQL（SQLite，内存数据库实现 Python + SQL 联动）</p></li></ul><h3>1.3 分析目标</h3><ol><li><p>使用 SQL 完成多维度用户画像分层统计（新老用户、年龄段 × 性别、流量渠道）；</p></li><li><p>搭建「首页 → 商品列表 → 商品详情 → 支付页 → 订单确认」五级转化漏斗，计算各环节转化率与流失率，定位流失卡点；</p></li><li><p>交叉分析用户属性与转化数据，识别高价值用户群体，输出运营优化建议。</p></li></ol><h3>1.4 简历项目描述</h3><blockquote><p><strong>电商用户行为画像与转化漏斗分析 | Python、SQL（SQLite）、Pandas、Matplotlib</strong></p><ul><li><p>基于约十万条电商用户行为数据，使用 Pandas 完成缺失值/异常值/重复值清洗与特征工程（转化标记、年龄段分组），清洗后保留 约5.8万条 用于分析，并借助 SQLite 实现 Python + SQL 联动的聚合分析；</p></li><li><p>构建五级购物转化漏斗，定位「商品详情页 → 支付页」为全链路最大流失环节，针对性提出支付流程简化与支付优惠券策略；</p></li><li><p>通过新老用户、流量渠道、年龄段 × 性别三维交叉分析，发现 12-24 岁女性用户转化率最高（8.02%）、老用户转化率约为新用户 2.4倍、Direct 渠道效率显著优于 SEO，输出渠道预算分配与精细化人群运营建议。</p></li></ul></blockquote><p></p>

<h2>二、数据导入与理解</h2><h3>2.1 加载工具库与数据</h3><p></p>

In [1]:
# ==========================================
# 2.1 加载工具库与数据
# ==========================================
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


In [2]:
#--------------中文字体设置-----------------------------
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")

环境准备完成


In [3]:
# ---------- 创建内存 SQLite 数据库，实现 Python + SQL 联动 ----------
conn = sqlite3.connect(':memory:')

def run_sql(sql_text):
    """执行 SQL 并返回 DataFrame，同时展示结果。"""
    result_df = pd.read_sql(sql_text, conn)
    display(result_df)
    return result_df

# ---------- 读取原始数据 ----------
df = pd.read_csv("用户行为分析.csv")

print(f"数据加载完成！原始数据形状：{df.shape}")
print(f"字段列表：{list(df.columns)}")


数据加载完成！原始数据形状：(100000, 13)
字段列表：['new_user', 'age', 'sex', 'market', 'device', 'operative_system', 'source', 'total_pages_visited', 'home_page', 'listing_page', 'product_page', 'payment_page', 'confirmation_page']


### 2.2 数据概览  

**字段说明**：  

| 字段 | 含义 | 类型 |  
|------|------|------|  
| sex | 用户性别 | 分类 |  
| age | 用户年龄 | 数值 |  
| new_user | 是否新用户（1=新，0=老） | 分类 |  
| source | 流量来源渠道 | 分类 |  
| home_page / listing_page / product_page / payment_page / confirmation_page | 用户在 首页 / 列表页 / 详情页 / 支付页 / 确认页 的访问次数 | 数值 |  


In [4]:
# ==========================================
# 2.2 数据概览
# ==========================================
print("=" * 50)
print("【数据基本信息】")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("【数据统计描述】")
print("=" * 50)
display(df.describe())

print("=" * 50)
print("【前5行样本】")
print("=" * 50)
display(df.head())


【数据基本信息】
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
new_user               100000 non-null int64
age                    100000 non-null int64
sex                    100000 non-null object
market                 100000 non-null int64
device                 100000 non-null object
operative_system       100000 non-null object
source                 100000 non-null object
total_pages_visited    100000 non-null int64
home_page              100000 non-null int64
listing_page           100000 non-null int64
product_page           100000 non-null int64
payment_page           100000 non-null int64
confirmation_page      100000 non-null int64
dtypes: int64(9), object(4)
memory usage: 9.9+ MB

【数据统计描述】


,new_user,age,market,total_pages_visited,home_page,listing_page,product_page,payment_page,confirmation_page
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.0,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.315250,30.39508,1.722380,5.907490,1.0,0.738940,0.496180,0.068620,0.024040
std,0.464618,8.04239,0.965203,4.199546,0.0,0.439215,0.499988,0.252808,0.153174
min,0.000000,17.00000,1.000000,1.000000,1.0,0.000000,0.000000,0.000000,0.000000
25%,0.000000,24.00000,1.000000,3.000000,1.0,0.000000,0.000000,0.000000,0.000000
50%,0.000000,30.00000,1.000000,5.000000,1.0,1.000000,0.000000,0.000000,0.000000
75%,1.000000,36.00000,2.000000,7.000000,1.0,1.000000,1.000000,0.000000,0.000000
max,1.000000,123.00000,4.000000,28.000000,1.0,1.000000,1.000000,1.000000,1.000000


【前5行样本】


,new_user,age,sex,market,device,operative_system,source,total_pages_visited,home_page,listing_page,product_page,payment_page,confirmation_page
0,0,17,Female,1,mobile,android,Direct,3,1,1,1,0,0
1,0,27,Male,2,mobile,iOS,Direct,5,1,0,0,0,0
2,0,24,Female,1,mobile,android,Seo,4,1,1,0,0,0
3,0,21,Male,2,desktop,windows,Direct,5,1,1,1,1,0
4,0,17,Male,2,mobile,iOS,Direct,6,1,1,0,0,0


## 三、数据清洗  

数据清洗是分析结论可信度的前提。本节按「**缺失值 → 异常值 → 重复值 → 数据类型 → 特征工程**」的标准流程处理，每一步均先检测、再处理、后验证，确保清洗过程可追溯。  


### 3.1 缺失值检测与处理

In [5]:
# ==========================================
# 3.1 缺失值检测与处理
# ==========================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'缺失数量': missing, '缺失比例(%)': missing_pct})

print("【缺失值统计】")
if missing_df['缺失数量'].sum() == 0:
    print("各字段均无缺失值，无需填充或剔除。")
else:
    display(missing_df[missing_df['缺失数量'] > 0])
print(f"总缺失值数量：{df.isnull().sum().sum()}")


【缺失值统计】
各字段均无缺失值，无需填充或剔除。
总缺失值数量：0


### 3.2 异常值识别与处理  

分类字段重点检查是否存在非法取值（如占位符 `0`）；数值字段重点检查取值范围是否符合业务常识（如年龄区间、页面访问次数非负）。  


In [6]:
# ==========================================
# 3.2 异常值识别与处理
# ==========================================
print("【分类字段异常值检测】")
print("\n1. 性别(sex)字段分布：")
print(df['sex'].value_counts(dropna=False))
print("\n2. 流量渠道(source)字段分布：")
print(df['source'].value_counts(dropna=False))
print("\n3. 新老用户(new_user)字段分布：")
print(df['new_user'].value_counts(dropna=False))

print("\n【数值字段异常值检测】")
print(f"4. 年龄(age)范围：{df['age'].min()} ~ {df['age'].max()}，"
      f"均值 {df['age'].mean():.1f}，中位数 {df['age'].median():.1f}")

page_cols = ['home_page', 'listing_page', 'product_page', 'payment_page', 'confirmation_page']
for col in page_cols:
    print(f"   {col} 负值数量：{(df[col] < 0).sum()}")

# ---------- 异常值处理：分类字段中的占位符 "0" 统一标记为 Unknown ----------
# 说明：将非法取值保留为 Unknown 而非直接删除，避免损失样本量
df['sex'] = df['sex'].replace(['0', 0], 'Unknown')
df['source'] = df['source'].replace(['0', 0], 'Unknown')
print("\n✅ 已将 sex / source 中的占位符 '0' 替换为 'Unknown'")


【分类字段异常值检测】

1. 性别(sex)字段分布：
Female    60554
Male      38673
0           773
Name: sex, dtype: int64

2. 流量渠道(source)字段分布：
Direct    56688
Seo       28337
Ads       14852
0           123
Name: source, dtype: int64

3. 新老用户(new_user)字段分布：
0    68475
1    31525
Name: new_user, dtype: int64

【数值字段异常值检测】
4. 年龄(age)范围：17 ~ 123，均值 30.4，中位数 30.0
   home_page 负值数量：0
   listing_page 负值数量：0
   product_page 负值数量：0
   payment_page 负值数量：0
   confirmation_page 负值数量：0

✅ 已将 sex / source 中的占位符 '0' 替换为 'Unknown'


<h3>3.3 重复值检查</h3><p>完全重复的行很可能是重复记录或重复上报，为避免同一行为被重复计数、影响转化率计算的准确性，予以删除。</p>

In [7]:
# ==========================================
# 3.3 重复值检查
# ==========================================
duplicate_count = df.duplicated().sum()
print("【重复行检测】")
print(f"完全重复的行数：{duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"已删除重复行，删除后数据形状：{df.shape}")
else:
    print("未发现重复行，数据干净。")


【重复行检测】
完全重复的行数：42006
已删除重复行，删除后数据形状：(57994, 13)


### 3.4 数据类型转换

In [8]:
# ==========================================
# 3.4 数据类型转换
# ==========================================
# 分类字段转换为 category 类型，显著降低内存占用并加速分组聚合
for col in ['sex', 'source', 'new_user']:
    df[col] = df[col].astype('category')

print("【数据类型转换完成】")
print(df.dtypes)


【数据类型转换完成】
new_user               category
age                       int64
sex                    category
market                    int64
device                   object
operative_system         object
source                 category
total_pages_visited       int64
home_page                 int64
listing_page              int64
product_page              int64
payment_page              int64
confirmation_page         int64
dtype: object


### 3.5 特征工程  

构造两个分析所需的关键特征：  

- **is_convert**：是否完成转化（以是否到达订单确认页为准），作为后续所有转化率计算的分子；  
- **age_group**：按业务场景将年龄划分为 12-24 岁青年 / 25-30 岁主力 / 31 岁以上成熟 三个客群。  


In [9]:
# ==========================================
# 3.5 特征工程
# ==========================================
# 1. 转化标记：访问过订单确认页即视为转化
df['is_convert'] = (df['confirmation_page'] > 0).astype(int)

# 2. 年龄段分组（pd.cut 分箱，与业务客群定义对齐）
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 24, 30, df['age'].max() + 1],
    labels=['12-24岁青年', '25-30岁主力', '31岁以上成熟'],
    right=True
)

print("【特征工程完成】新增字段：is_convert, age_group")
print(f"清洗后数据形状：{df.shape}")

# ---------- 清洗后的数据写入 SQLite，供后续 SQL 分析使用 ----------
df.to_sql("user_behavior", conn, index=False, if_exists="replace")
print("\n✅ 数据已导入 SQLite，表名：user_behavior")
run_sql("SELECT * FROM user_behavior LIMIT 5;")


【特征工程完成】新增字段：is_convert, age_group
清洗后数据形状：(57994, 15)

✅ 数据已导入 SQLite，表名：user_behavior


,new_user,age,sex,market,device,operative_system,source,total_pages_visited,home_page,listing_page,product_page,payment_page,confirmation_page,is_convert,age_group
0,0,17,Female,1,mobile,android,Direct,3,1,1,1,0,0,0,12-24岁青年
1,0,27,Male,2,mobile,iOS,Direct,5,1,0,0,0,0,0,25-30岁主力
2,0,24,Female,1,mobile,android,Seo,4,1,1,0,0,0,0,12-24岁青年
3,0,21,Male,2,desktop,windows,Direct,5,1,1,1,1,0,0,12-24岁青年
4,0,17,Male,2,mobile,iOS,Direct,6,1,1,0,0,0,0,12-24岁青年


,new_user,age,sex,market,device,operative_system,source,total_pages_visited,home_page,listing_page,product_page,payment_page,confirmation_page,is_convert,age_group
0,0,17,Female,1,mobile,android,Direct,3,1,1,1,0,0,0,12-24岁青年
1,0,27,Male,2,mobile,iOS,Direct,5,1,0,0,0,0,0,25-30岁主力
2,0,24,Female,1,mobile,android,Seo,4,1,1,0,0,0,0,12-24岁青年
3,0,21,Male,2,desktop,windows,Direct,5,1,1,1,1,0,0,12-24岁青年
4,0,17,Male,2,mobile,iOS,Direct,6,1,1,0,0,0,0,12-24岁青年


## 四、用户画像分析  

### 4.1 新老用户转化分析  

通过 SQL 聚合统计新、老用户的整体转化数据，对比两类用户的转化水平，判断平台的新客承接能力与新客/老客的价值差异。  


In [10]:
# ==========================================
# 4.1 新老用户转化分析
# ==========================================
sql_new_old = """
SELECT
    new_user,
    COUNT(*) AS 用户总数量,
    SUM(is_convert) AS 成功转化人数,
    ROUND(SUM(is_convert) * 1.0 / COUNT(*), 4) AS 整体转化率
FROM user_behavior
GROUP BY new_user;
"""
new_old_result = run_sql(sql_new_old)

# 映射中文标签（用字典取值，避免依赖行序）
new_old_result['new_user'] = new_old_result['new_user'].astype(int)
new_old_result['用户类型'] = new_old_result['new_user'].map({0: '老用户', 1: '新用户'})
rate_map = dict(zip(new_old_result['用户类型'], new_old_result['整体转化率']))

# ------------------绘图 --------------------
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(new_old_result['用户类型'], new_old_result['整体转化率'],
       color=['#2E86AB', '#A2B372'], width=0.5, edgecolor='white')

for bar, rate in zip(bars, new_old_result['整体转化率']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
        f'{rate * 100:.2f}%', ha='center', va='bottom',
        fontsize=12, fontweight='bold')

ax.set_title('新老用户转化率对比', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('转化率', fontsize=12)

for label in ax.get_xticklabels():
    label.set_fontsize(12)

ax.set_ylim(0, new_old_result['整体转化率'].max() * 1.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ------------- 结论文本 ---------------------
print(f"\n【结论】老用户转化率 {rate_map['老用户'] * 100:.2f}%，新用户转化率 {rate_map['新用户'] * 100:.2f}%，"
      f"老用户转化能力约为新用户的 {rate_map['老用户'] / rate_map['新用户']:.1f} 倍。")


,new_user,用户总数量,成功转化人数,整体转化率
0,0,35387,1734,0.0490
1,1,22607,457,0.0202


<Figure size 576x360 with 1 Axes>


【结论】老用户转化率 4.90%，新用户转化率 2.02%，老用户转化能力约为新用户的 2.4 倍。


### 4.2 流量渠道转化分析  

统计各流量渠道的用户规模与转化能力，识别高价值渠道与低效渠道，为投放预算分配提供数据依据。  


In [11]:
# ==========================================
# 4.2 流量渠道转化分析
# ==========================================
sql_source = """
SELECT
    source AS 流量渠道,
    COUNT(*) AS 访问总用户,
    SUM(is_convert) AS 下单转化用户,
    ROUND(SUM(is_convert) * 1.0 / COUNT(*), 4) AS 渠道转化率
FROM user_behavior
GROUP BY source
ORDER BY 渠道转化率 DESC;
"""
source_df = run_sql(sql_source)

# ------------------绘图 --------------------------
fig, ax = plt.subplots(figsize=(10, 6))
palette = ['#2E86AB', '#A2B372', '#F18F01', '#C73E1D', '#5BC0BE']
colors = [palette[i % len(palette)] for i in range(len(source_df))]
bars = ax.barh(source_df['流量渠道'], source_df['渠道转化率'], color=colors, edgecolor='white')

for bar, rate in zip(bars, source_df['渠道转化率']):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height() / 2,
        f'{rate * 100:.2f}%', ha='left', va='center',
        fontsize=11)

ax.set_title('各流量渠道转化率排名', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('转化率', fontsize=12)

for label in ax.get_yticklabels():
    label.set_fontsize(12)

ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ----------------- 结论文本 -----------------------
best, worst = source_df.iloc[0], source_df.iloc[-1]
print(f"\n【结论】转化率最高渠道：{best['流量渠道']}（{best['渠道转化率'] * 100:.2f}%），"
      f"最低渠道：{worst['流量渠道']}（{worst['渠道转化率'] * 100:.2f}%），渠道效率差异明显。")


,流量渠道,访问总用户,下单转化用户,渠道转化率
0,Direct,26444,1514,0.0573
1,Seo,19717,432,0.0219
2,Ads,11711,243,0.0207
3,Unknown,122,2,0.0164


<Figure size 720x432 with 1 Axes>


【结论】转化率最高渠道：Direct（5.73%），最低渠道：Unknown（1.64%），渠道效率差异明显。


### 4.3 年龄与性别交叉分析  

通过年龄段与性别双维度分组聚合，对比不同人群的转化表现，定位平台核心高价值客群，为定向营销活动提供人群筛选依据。  


In [12]:
# ==========================================
# 4.3 年龄与性别交叉分析
# ==========================================
sql_age_sex = """
SELECT
    age_group AS 年龄段,
    sex AS 用户性别,
    COUNT(*) AS 总用户数,
    SUM(is_convert) AS 转化用户数,
    ROUND(SUM(is_convert) * 1.0 / COUNT(*), 4) AS 转化率
FROM user_behavior
GROUP BY age_group, sex
ORDER BY age_group, sex;
"""
age_sex_df = run_sql(sql_age_sex)

# --------绘图 --------
fig, ax = plt.subplots(figsize=(12, 6))
markers = ['o', 's', '^', 'D']
for i, gender in enumerate(age_sex_df['用户性别'].unique()):
    data = age_sex_df[age_sex_df['用户性别'] == gender]
    ax.plot(data['年龄段'].astype(str), data['转化率'],
        marker=markers[i % len(markers)], linewidth=2, label=gender)

ax.set_title('各年龄段男/女用户转化率对比', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('年龄段', fontsize=12)
ax.set_ylabel('转化率', fontsize=12)

for label in ax.get_xticklabels():
    label.set_fontsize(12)

legend = ax.legend(title='性别')
for text in legend.get_texts():
    text.set_fontsize(11)
legend.get_title().set_fontsize(12)

ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# ---------- 结论文本 ----------
top_group = age_sex_df.loc[age_sex_df['转化率'].idxmax()]
print(f"\n【结论】转化率最高客群：{top_group['年龄段']} · {top_group['用户性别']}用户，"
      f"转化率 {top_group['转化率'] * 100:.2f}%，是平台最具价值的人群。")


,年龄段,用户性别,总用户数,转化用户数,转化率
0,12-24岁青年,Female,8586,689,0.0802
1,12-24岁青年,Male,6724,216,0.0321
2,12-24岁青年,Unknown,185,6,0.0324
3,25-30岁主力,Female,8238,508,0.0617
4,25-30岁主力,Male,5806,171,0.0295
5,25-30岁主力,Unknown,201,6,0.0299
6,31岁以上成熟,Female,16031,438,0.0273
7,31岁以上成熟,Male,11859,154,0.0130
8,31岁以上成熟,Unknown,364,3,0.0082


<Figure size 864x432 with 1 Axes>


【结论】转化率最高客群：12-24岁青年 · Female用户，转化率 8.02%，是平台最具价值的人群。


## 五、转化漏斗分析  

### 5.1 全链路转化漏斗构建  

利用 Pandas 统计首页、商品列表、商品详情、支付页、订单确认页 5 个环节的独立访客数，绘制居中漏斗图，量化各环节的转化与流失规模。  


In [13]:
# ==========================================
# 5.1 全链路转化漏斗构建
# ==========================================
stages = ['首页', '商品列表', '商品详情', '支付页面', '订单确认']
page_map = dict(zip(stages, page_cols))
user_nums = [df[df[col] > 0].shape[0] for col in page_cols]

# 环节转化率 / 流失率（相对上一环节）
conv_rates = [1.0] + [user_nums[i] / user_nums[i - 1] for i in range(1, len(user_nums))]
loss_rates = [1 - r for r in conv_rates[1:]]
overall_rate = user_nums[-1] / user_nums[0]

# -------- 居中漏斗图 --------
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1F77B4', '#419FD9', '#5BC0BE', '#F1C40F', '#E74C3C']
max_num = user_nums[0]

for i, (stage, num) in enumerate(zip(stages, user_nums)):      # 模拟漏斗自上而下收窄
    left = (max_num - num) / 2
    ax.barh(i, num, left=left, height=0.6, color=colors[i], edgecolor='white')
    ax.text(max_num / 2, i, f'{stage}  {num:,}',
        ha='center', va='center', fontsize=11,
        color='#1A202C', fontweight='bold')
    if i > 0:
        ax.text(max_num + max_num * 0.02, i - 0.5,
            f'转化 {conv_rates[i] * 100:.1f}%（流失 {loss_rates[i - 1] * 100:.1f}%）',
            ha='left', va='center', fontsize=10,
            color='#C73E1D')

ax.set_title(f'电商全链路转化漏斗（整体转化率 {overall_rate * 100:.2f}%）', fontsize=10, fontweight='bold', pad=15)
ax.set_xlim(0, max_num * 1.35)
ax.set_ylim(len(stages) - 0.5, -0.5)    # 顶部为首页
ax.axis('off')                         # 隐藏坐标轴,突出漏斗形态
plt.tight_layout()
plt.show()

# ---------- 打印详细数据 ----------
print("\n【漏斗各环节数据】")
for i, (stage, num) in enumerate(zip(stages, user_nums)):
    if i == 0:
        print(f'{stage}: {num:,}人（基准）')
    else:
        print(f'{stage}: {num:,}人（环节转化 {conv_rates[i] * 100:.1f}%，流失 {loss_rates[i - 1] * 100:.1f}%）')
print(f'\n整体转化率（首页 → 订单确认）：{overall_rate * 100:.2f}%')


<Figure size 720x432 with 1 Axes>


【漏斗各环节数据】
首页: 57,994人（基准）
商品列表: 41,330人（环节转化 71.3%，流失 28.7%）
商品详情: 26,209人（环节转化 63.4%，流失 36.6%）
支付页面: 5,770人（环节转化 22.0%，流失 78.0%）
订单确认: 2,191人（环节转化 38.0%，流失 62.0%）

整体转化率（首页 → 订单确认）：3.78%


### 5.2 漏斗流失诊断  

按流失率对各环节排序，定位全链路最大流失卡点，为优化优先级排序提供依据。  


In [14]:
# ==========================================
# 5.2 漏斗流失诊断
# ==========================================
stage_pairs = [
    ('首页 → 商品列表', loss_rates[0]),
    ('商品列表 → 商品详情', loss_rates[1]),
    ('商品详情 → 支付页面', loss_rates[2]),
    ('支付页面 → 订单确认', loss_rates[3]),
]
sorted_pairs = sorted(stage_pairs, key=lambda x: x[1], reverse=True)

print("【各环节流失严重程度排名】")
for rank, (pair, rate) in enumerate(sorted_pairs, 1):
    severity = '\U0001F534 严重' if rate > 0.5 else ('\U0001F7E1 中等' if rate > 0.3 else '\U0001F7E2 轻微')
    print(f'{rank}. {pair}: 流失率 {rate * 100:.1f}%  {severity}')

worst_stage = sorted_pairs[0]
print(f'\n【核心发现】流失最严重的环节是「{worst_stage[0]}」，流失率高达 {worst_stage[1] * 100:.1f}%，'
      '应作为转化率优化的第一优先级。')


【各环节流失严重程度排名】
1. 商品详情 → 支付页面: 流失率 78.0%  🔴 严重
2. 支付页面 → 订单确认: 流失率 62.0%  🔴 严重
3. 商品列表 → 商品详情: 流失率 36.6%  🟡 中等
4. 首页 → 商品列表: 流失率 28.7%  🟢 轻微

【核心发现】流失最严重的环节是「商品详情 → 支付页面」，流失率高达 78.0%，应作为转化率优化的第一优先级。


<h2>六、结论与业务优化建议</h2><h3>6.1 核心结论</h3><ol><li><p><strong>转化漏斗</strong>：「商品详情页 → 支付页面」是全链路流失最严重的环节，大量用户在浏览商品后放弃支付，支付体验是转化率提升的第一杠杆；</p></li><li><p><strong>渠道投放</strong>：Direct（直接访问）渠道转化率最高，SEO 渠道转化率最低，渠道间效率差异显著，预算分配应与渠道转化能力匹配；</p></li><li><p><strong>用户画像</strong>：12-24 岁青年女性用户转化率最高（8.02%），是平台最具价值的客群；老用户转化能力约为新用户的 2.4 倍，用户留存价值突出。</p></li></ol><h3>6.2 落地运营优化建议</h3><table style="min-width: 75px;"><colgroup><col style="min-width: 25px;"><col style="min-width: 25px;"><col style="min-width: 25px;"></colgroup><tbody><tr><th colspan="1" rowspan="1"><p>优化方向</p></th><th colspan="1" rowspan="1"><p>具体措施</p></th><th colspan="1" rowspan="1"><p>预期效果</p></th></tr><tr><td colspan="1" rowspan="1"><p><strong>支付体验优化</strong></p></td><td colspan="1" rowspan="1"><p>简化支付页操作流程，接入多种支付方式，上线支付环节专属优惠券</p></td><td colspan="1" rowspan="1"><p>降低详情页 → 支付页流失率，直接拉升整体转化</p></td></tr><tr><td colspan="1" rowspan="1"><p><strong>渠道投放优化</strong></p></td><td colspan="1" rowspan="1"><p>提升 Direct 渠道用户维护投入，优化 SEO 落地页与搜索词的匹配质量，缩减低效渠道预算</p></td><td colspan="1" rowspan="1"><p>提高投放 ROI，让预算向高转化渠道倾斜</p></td></tr><tr><td colspan="1" rowspan="1"><p><strong>精准用户运营</strong></p></td><td colspan="1" rowspan="1"><p>针对 12-24 岁女性用户推送美妆/服饰类专属活动；针对新用户发放首单优惠券</p></td><td colspan="1" rowspan="1"><p>放大高价值客群价值，补齐新客转化短板</p></td></tr><tr><td colspan="1" rowspan="1"><p><strong>老用户召回</strong></p></td><td colspan="1" rowspan="1"><p>建立积分体系与会员等级，通过短信/Push 推送个性化商品推荐</p></td><td colspan="1" rowspan="1"><p>巩固老用户 2.4 倍转化优势，提升复购率</p></td></tr></tbody></table><h3>6.3 项目复盘</h3><ul><li><p><strong>方法论层面</strong>：本项目遵循「数据清洗 → 描述性统计 → 多维交叉分析 → 漏斗诊断 → 业务建议」的完整分析闭环，SQL 与 Pandas 各司其职（SQL 负责聚合、Pandas 负责清洗与可视化）；</p></li><li><p><strong>可拓展方向</strong>：引入时间维度做留存/复购分析、基于 RFM 模型做用户分层、对漏斗各环节做 A/B 测试验证优化效果。</p></li></ul><p></p>